# Experiment 1 · 01 — Consensus labels for the 4,349 report-only studies

**Why:** every public 0.941–0.943 model was trained on the older report-label tables (about 0.87 macro AUC against
the 58 image-labelled studies). The single models that reach 0.949–0.950 got there with better labels, not
bigger networks (forum 735304). This notebook builds those labels.

**What it does:**
1. Finds every attached report-label table (any CSV/parquet with a StudyInstanceUID column and the 12 findings).
2. Scores each one on the 58 gold studies: macro ROC-AUC with a 2,000-sample bootstrap CI.
3. Builds a consensus from the best sources, using pre-registered rules and not per-target cherry-picking.
4. Applies two evidence-backed rules for findings the report doesn't mention (forum 733932).
   Each rule is kept only if it doesn't lower its own target's gold AUC.
5. Writes `labels_v1.parquet` + `folds_v1.json` (5 folds grouped by identical report text) for notebook 02.

CPU only · internet off · about 2–4 minutes. The gold studies are **never** used as training labels anywhere in this experiment.

In [ ]:
import os, re, json, hashlib, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA", "PF OA",
           "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
INPUT_ROOT = os.environ.get("NX_INPUT_ROOT", "/kaggle/input")
OUT = os.environ.get("NX_OUT", "/kaggle/working")
SEED, N_BOOT, N_FOLDS = 2026, 2000, 5
MIN_COVERAGE = 0.90     # a source must label >= 90% of the report-only studies
ACCEPT_MARGIN = 0.03    # accept sources within 0.03 gold macro AUC of the best source
SILENCE_CAP = 0.10      # Baker's / Medial OA not mentioned -> at most 0.10 (gold positive rate 0.03 / 0.00 when silent)
# "consensus": always train on the consensus (decided 2026-09-24 after run 1: the gate's best single table, tsuyu122,
#              has unclear provenance; the consensus scored 0.9205 vs 0.9124 with P(better)=0.83 - see runs/RUN_LOG.md)
# "gate":      follow the pre-registered G1 rule
LABEL_CHOICE = "consensus"
rng = np.random.default_rng(SEED)
os.makedirs(OUT, exist_ok=True)

## 1 · Competition data and the 58 gold studies

In [ ]:
def find_competition_root():
    for c in (f"{INPUT_ROOT}/competitions/rsna-knee-abnormality-detection", f"{INPUT_ROOT}/rsna-knee-abnormality-detection"):
        if os.path.isfile(f"{c}/train.csv"):
            return c
    for d, dirs, fs in os.walk(INPUT_ROOT):
        dirs[:] = [x for x in dirs if not x.endswith(("_series", "_images"))]
        if "train.csv" in fs and ("test.csv" in fs or "train_series.csv" in fs):
            return d
    raise FileNotFoundError("competition train.csv not found - attach the competition data")

ROOT = find_competition_root()
train = pd.read_csv(f"{ROOT}/train.csv", dtype={"StudyInstanceUID": str})
gold_mask = train[TARGETS].notna().all(axis=1)
gold = train[gold_mask].set_index("StudyInstanceUID")[TARGETS].astype(float)
rest_ids = train.loc[~gold_mask, "StudyInstanceUID"].tolist()
print(f"train {len(train)} | gold {len(gold)} | report-only {len(rest_ids)}")
print("gold positives:", gold.sum().astype(int).to_dict())

## 2 · Discover and normalise every attached label table

In [ ]:
def _norm(s):
    return re.sub(r"[^a-z]", "", str(s).lower())

_KEY = {_norm(t): t for t in TARGETS}                       # 'bakers' -> "Baker's"
_TXT = {"yes": 1.0, "y": 1.0, "true": 1.0, "positive": 1.0, "pos": 1.0, "present": 1.0,
        "no": 0.0, "n": 0.0, "false": 0.0, "negative": 0.0, "neg": 0.0, "absent": 0.0,
        "unk": np.nan, "unknown": np.nan, "na": np.nan, "nan": np.nan, "none": np.nan, "": np.nan,
        "notaddressed": np.nan, "uncertain": np.nan, "notmentioned": np.nan}
_BAD = ("conf", "weight", "state", "provenance", "source", "reason", "evidence", "std", "var", "mask", "fold")
# helper files shipped next to label tables: sample weights, confidences, per-cell states, gold benchmarks, submissions
_BAD_FILES = ("weight", "conf", "state", "provenance", "gold", "submission", "test")

def map_columns(cols):
    """Map a table's columns to the 12 targets; confidence/weight/state columns are ignored."""
    out = {}
    for c in cols:
        n = _norm(c)
        if any(b in n for b in _BAD):
            continue
        for suf in ("", "prob", "p", "score", "soft", "label", "pred", "y"):
            base = n[:len(n) - len(suf)] if suf else n
            if (not suf or n.endswith(suf)) and base in _KEY:
                out.setdefault(_KEY[base], c)
                break
    return out

def coerce(col):
    """YES/NO/UNK words -> 1/0/NaN, numbers -> float."""
    if not pd.api.types.is_numeric_dtype(col):      # object (pandas 2) or str (pandas 3) columns
        s = col.astype(str).str.strip().str.lower().str.replace(r"[\s_\-']", "", regex=True)
        word = s.isin(_TXT.keys())
        return s.map(_TXT).where(word, pd.to_numeric(s, errors="coerce")).astype(float)
    return pd.to_numeric(col, errors="coerce").astype(float)

def discover_sources():
    found, log = {}, []
    for d, dirs, fs in os.walk(INPUT_ROOT):
        dirs[:] = [x for x in dirs if not x.endswith(("_series", "_images"))]
        if os.path.abspath(d).startswith(os.path.abspath(ROOT)):
            continue
        for f in sorted(fs):
            if not f.endswith((".csv", ".parquet")) or any(x in f.lower() for x in _BAD_FILES):
                continue
            p = os.path.join(d, f)
            try:
                df = pd.read_parquet(p) if f.endswith(".parquet") else pd.read_csv(p, low_memory=False)
            except Exception as e:
                log.append(("unreadable", p, str(e)[:80])); continue
            uid = next((c for c in df.columns if _norm(c) in ("studyinstanceuid", "studyuid", "studyid")), None)
            cm = map_columns([c for c in df.columns if c != uid])
            if uid is None or len(cm) < 10:
                continue
            df[uid] = df[uid].astype(str)
            df = df.drop_duplicates(uid).set_index(uid)
            vals = pd.DataFrame({t: coerce(df[cm[t]]) if t in cm else np.nan for t in TARGETS}, index=df.index)
            finite = vals.to_numpy()[np.isfinite(vals.to_numpy())]
            if finite.size == 0 or finite.min() < -1e-6 or finite.max() > 1 + 1e-6:
                log.append(("values outside [0,1]", p, f"{finite.min() if finite.size else 'empty'}")); continue
            name = f"{os.path.basename(d)}/{os.path.splitext(f)[0]}"
            found[name] = vals.reindex(train["StudyInstanceUID"])
    return found, log

SOURCES, SKIPPED = discover_sources()
print(f"{len(SOURCES)} candidate label tables")
for s in SKIPPED:
    print("  skipped:", *s)

## 3 · Score every source on the 58 gold studies (bootstrap CI)

In [ ]:
def auc(y, p):
    """Mann-Whitney ROC-AUC with average ranks for ties (NaN if one class is missing)."""
    y = np.asarray(y) > 0.5
    n1 = int(y.sum()); n0 = len(y) - n1
    if n1 == 0 or n0 == 0:
        return np.nan
    r = pd.Series(np.asarray(p, float)).rank(method="average").to_numpy()
    return (r[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

G_IDS = gold.index.tolist()
GY = gold.to_numpy()
BOOT = [rng.integers(0, len(G_IDS), len(G_IDS)) for _ in range(N_BOOT)]

def per_target(pred):
    P = pred.reindex(G_IDS).to_numpy(float)
    out = {}
    for j, t in enumerate(TARGETS):
        m = np.isfinite(P[:, j])
        out[t] = auc(GY[m, j], P[m, j]) if m.sum() >= 10 else np.nan
    return out

def boot_macro(pred):
    P = pred.reindex(G_IDS).to_numpy(float)
    vals = []
    for ii in BOOT:
        a = []
        for j in range(len(TARGETS)):
            p, y = P[ii, j], GY[ii, j]
            m = np.isfinite(p)
            v = auc(y[m], p[m]) if m.sum() >= 10 else np.nan
            if np.isfinite(v):
                a.append(v)
        vals.append(np.mean(a) if a else np.nan)
    return np.asarray(vals)

def coverage(pred):
    return float((pred.reindex(rest_ids).notna().sum(axis=1) >= 1).mean())   # UNK cells are gaps, not missing rows

rows = []
for name, pred in SOURCES.items():
    pt = per_target(pred)
    rows.append({"source": name, "dataset": name.split("/")[0], "coverage": coverage(pred),
                 "gold_macro": float(np.nanmean(list(pt.values()))), **{f"auc_{t}": pt[t] for t in TARGETS}})
bench = pd.DataFrame(rows).sort_values("gold_macro", ascending=False).reset_index(drop=True)
# a table that reproduces the gold labels is a copy of train.csv, not a report teacher
bench["gold_copy"] = (bench["gold_macro"] > 0.995) | bench["source"].str.contains("gold", case=False)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
print(bench[["source", "coverage", "gold_macro", "gold_copy"]].to_string())

## 4 · Consensus (pre-registered rules)
* **Eligible:** coverage ≥ 90%, not a gold copy, **one table per dataset** (the best one), so that three versions
  of the same author's labels don't outvote everyone else.
* **Accepted:** eligible and within 0.03 gold macro AUC of the best eligible table.
* **Consensus:** per-cell mean of the accepted tables' probabilities. This keeps them soft for the BCE loss.

In [ ]:
elig = bench[(bench.coverage >= MIN_COVERAGE) & ~bench.gold_copy].drop_duplicates("dataset")
best_macro = float(elig.gold_macro.max())
accepted = elig[elig.gold_macro >= best_macro - ACCEPT_MARGIN].source.tolist()
best_single = elig.iloc[0].source
print("eligible:", elig.source.tolist())
print("accepted:", accepted, "| best single:", best_single, f"({best_macro:.4f})")

stack = np.stack([SOURCES[s].to_numpy(float) for s in accepted])      # (S, N, 12)
C0 = pd.DataFrame(np.nanmean(stack, axis=0), index=train["StudyInstanceUID"], columns=TARGETS)

## 5 · Findings the report doesn't mention (forum 733932)
stevenleehans codes "report doesn't address this finding" as exactly 0.5. How often gold is positive:
* **silent vs mentioned:** Baker's 0.03 vs 0.44, Medial OA 0.00 vs 0.36. Silence ≈ negative → cap at 0.10.
* **Synovitis:** 0.34 vs 0.76. Silence says nothing, and the Effusion statement predicts gold Synovitis better
  → fill unaddressed Synovitis from the Effusion consensus.

The indicator comes from the accepted table with the most exact-0.5 Synovitis cells, if that share is above 30%.
Each rule is kept only if its own target's gold AUC does not drop.

In [ ]:
share05 = {s: float((SOURCES[s]["Synovitis"].reindex(rest_ids) == 0.5).mean()) for s in SOURCES}
ind_src = max(share05, key=share05.get) if share05 else None
decisions = {"silence_indicator": ind_src, "silence_share_synovitis": share05.get(ind_src)}
C = C0.copy()
if ind_src and share05[ind_src] > 0.30:
    silent = SOURCES[ind_src].eq(0.5)
    # rule A: Synovitis
    CA = C.copy()
    mA = silent["Synovitis"] & C["Effusion"].notna()
    CA.loc[mA, "Synovitis"] = C.loc[mA, "Effusion"]
    a0, a1 = per_target(C)["Synovitis"], per_target(CA)["Synovitis"]
    decisions["synovitis_from_effusion"] = {"cells": int(mA.sum()), "gold_before": a0, "gold_after": a1, "kept": bool(a1 >= a0)}
    if a1 >= a0:
        C = CA
    # rule B: Baker's and Medial OA
    for t in ("Baker's", "Medial OA"):
        CB = C.copy()
        mB = silent[t]
        CB.loc[mB, t] = np.minimum(C.loc[mB, t], SILENCE_CAP)
        b0, b1 = per_target(C)[t], per_target(CB)[t]
        decisions[f"silence_cap_{t}"] = {"cells": int(mB.sum()), "gold_before": b0, "gold_after": b1, "kept": bool(b1 >= b0)}
        if b1 >= b0:
            C = CB
else:
    decisions["note"] = "no table codes silence as 0.5 - silence rules skipped"
print(json.dumps(decisions, indent=1, default=float))

## 6 · Gate G1: consensus vs the best single table

In [ ]:
cons_pt, single_pt = per_target(C), per_target(SOURCES[best_single])
cons_macro, single_macro = float(np.nanmean(list(cons_pt.values()))), float(np.nanmean(list(single_pt.values())))
bc, bs = boot_macro(C), boot_macro(SOURCES[best_single])
worse = [t for t in TARGETS if np.isfinite(cons_pt[t]) and np.isfinite(single_pt[t]) and cons_pt[t] < single_pt[t] - 1e-9]
g1 = (cons_macro >= single_macro) and len(worse) <= 3
print(f"consensus {cons_macro:.4f} [{np.nanpercentile(bc, 2.5):.4f}, {np.nanpercentile(bc, 97.5):.4f}]  vs  "
      f"best single {single_macro:.4f} [{np.nanpercentile(bs, 2.5):.4f}, {np.nanpercentile(bs, 97.5):.4f}]")
print(f"P(consensus > single) over paired bootstrap: {np.nanmean(bc > bs):.2f} | worse on {len(worse)} targets: {worse}")
print("G1", "PASS" if g1 else "FAIL", "| LABEL_CHOICE =", LABEL_CHOICE)
use_consensus = g1 or LABEL_CHOICE == "consensus"
print("training labels ->", "consensus" if use_consensus else f"best single table ({best_single})")
FINAL = C if use_consensus else SOURCES[best_single].copy()
per_t = pd.DataFrame({"consensus": cons_pt, "best_single": single_pt}).round(4)
print(per_t.to_string())

## 7 · Fill gaps, write labels and grouped folds

In [ ]:
L = FINAL.reindex(rest_ids).copy()
n_missing = int(L.isna().sum().sum())
L = L.fillna(L.median())                      # rare: cells no accepted table covers
L = L.clip(0.0, 1.0).astype(np.float32)
L.index.name = "StudyInstanceUID"
L.reset_index().to_parquet(f"{OUT}/labels_v1.parquet", index=False)
for tag, frame in (("consensus", C), ("best_single", SOURCES[best_single])):   # both kept for later experiments
    alt = frame.reindex(rest_ids).fillna(frame.reindex(rest_ids).median()).clip(0, 1).astype(np.float32)
    alt.index.name = "StudyInstanceUID"
    alt.reset_index().to_parquet(f"{OUT}/labels_{tag}.parquet", index=False)

# folds: identical (normalised) report text never crosses folds; stratified on rare findings
rep = train.set_index("StudyInstanceUID").loc[rest_ids, "Report"].fillna("").astype(str)
grp = rep.str.lower().str.replace(r"\s+", " ", regex=True).str.strip().map(lambda s: hashlib.sha1(s.encode()).hexdigest())
strata = ((L["ACL"] > .5).astype(int) * 8 + (L["Fracture"] > .5).astype(int) * 4
          + ((L["Medial Meniscus"] > .5) | (L["Lateral Meniscus"] > .5)).astype(int) * 2 + (L["Effusion"] > .5).astype(int))
g_df = pd.DataFrame({"grp": grp, "stratum": strata}).groupby("grp").stratum.max().reset_index()
fold_of_group, k = {}, 0
for _, part in g_df.groupby("stratum"):
    groups = part.grp.sort_values().tolist()
    rng.shuffle(groups)
    for gname in groups:                      # one running counter across strata keeps folds balanced
        fold_of_group[gname] = k % N_FOLDS
        k += 1
folds = {u: int(fold_of_group[grp[u]]) for u in rest_ids}
json.dump({"folds": folds, "n_folds": N_FOLDS, "grouping": "sha1(normalised report text)", "seed": SEED},
          open(f"{OUT}/folds_v1.json", "w"))
bench.to_csv(f"{OUT}/label_benchmark.csv", index=False)
report = {"n_report_only": len(rest_ids), "n_gold": len(gold), "accepted": accepted, "best_single": best_single,
          "g1_pass": bool(g1), "label_choice": LABEL_CHOICE, "labels_v1_is": "consensus" if use_consensus else best_single,
          "consensus_macro": cons_macro, "best_single_macro": single_macro,
          "p_consensus_better": float(np.nanmean(bc > bs)), "per_target": per_t.to_dict(),
          "decisions": decisions, "filled_missing_cells": n_missing,
          "fold_sizes": pd.Series(folds).value_counts().sort_index().to_dict(),
          "label_means": L.mean().round(4).to_dict()}
json.dump(report, open(f"{OUT}/labels_v1_report.json", "w"), indent=1, default=float)
print(f"wrote labels_v1.parquet {L.shape}, folds_v1.json, label_benchmark.csv, labels_v1_report.json "
      f"(filled {n_missing} missing cells)")
print("fold sizes:", report["fold_sizes"])